# Project: Investigate data

**Data set:** `movies.csv` (main table) and `genre_groups.csv` (second table), from the vega-datasets collection.

**Environment:** Python 3.14.10, pandas 3.0.5, matplotlib 3.11.2.

Loosly formed plan: understand, clean, merge, describe, draw, answer, check the source.

In [ ]:
import sys
import pandas as pd
import numpy as np
import matplotlib as plt

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.__version__)

Python: 3.14.10
Pandas: 3.0.5
Matplotlib: 3.11.2


## Part 1: Read and understand the data

In [ ]:
movies_raw = pd.read_csv("data/movies.csv", parse_dates=["Release Date"], date_format="%b %d %Y")
genres = pd.read_csv("data/genre_groups.csv")

movies = movies_raw.copy()

print(f"{movies.shape[0]} rows, {movies.shape[1]} columns")

display(movies.dtypes)

display(movies.sample(10, random_state=1))

3201 rows, 16 columns


Title                                str
US Gross                         float64
Worldwide Gross                  float64
US DVD Sales                     float64
Production Budget                float64
Release Date              datetime64[us]
MPAA Rating                          str
Running Time min                 float64
Distributor                          str
Source                               str
Major Genre                          str
Creative Type                        str
Director                             str
Rotten Tomatoes Rating           float64
IMDB Rating                      float64
IMDB Votes                       float64
dtype: object

,Title,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,MPAA Rating,Running Time min,Distributor,Source,Major Genre,Creative Type,Director,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
1751,Firestorm,8123860.0,8123860.0,NaN,19000000.0,1998-01-09,R,NaN,20th Century Fox,Original Screenplay,Action,Contemporary Fiction,NaN,NaN,4.4,2118.0
909,Star Trek: Generations,75671262.0,120000000.0,NaN,38000000.0,1994-11-18,PG,NaN,Paramount Pictures,Based on TV,Adventure,Science Fiction,NaN,NaN,6.5,26465.0
979,Trois,1161843.0,1161843.0,NaN,200000.0,2000-02-11,NC-17,NaN,Rainforest Productions,Original Screenplay,Thriller/Suspense,Contemporary Fiction,NaN,NaN,3.3,360.0
329,Fled,17192205.0,19892205.0,NaN,30000000.0,1996-07-19,R,98.0,MGM,Original Screenplay,Action,Contemporary Fiction,NaN,14.0,4.9,4215.0
1885,Ghost World,6217849.0,8764007.0,NaN,5500000.0,2001-07-20,R,NaN,MGM,Based on Comic/Graphic Novel,Comedy,Contemporary Fiction,Terry Zwigoff,92.0,7.7,42973.0
1967,How to Deal,14108518.0,14108518.0,NaN,16000000.0,2003-07-18,PG-13,NaN,New Line,Based on Book/Short Story,Drama,Contemporary Fiction,NaN,29.0,5.4,5292.0
2131,K-PAX,50315140.0,50315140.0,NaN,48000000.0,2001-10-26,PG-13,121.0,Universal,Based on Book/Short Story,Drama,Science Fiction,Iain Softley,40.0,7.3,50475.0
98,Beverly Hills Cop III,42586861.0,119180938.0,NaN,50000000.0,1994-05-25,R,NaN,Paramount Pictures,Original Screenplay,Action,Contemporary Fiction,John Landis,10.0,5.0,21199.0
340,Forrest Gump,329694499.0,679400525.0,NaN,55000000.0,1994-07-06,PG-13,NaN,Paramount Pictures,Based on Book/Short Story,Drama,Historical Fiction,Robert Zemeckis,70.0,8.6,300455.0
1012,The Work and the Glory,3347647.0,3347647.0,NaN,7500000.0,2004-11-24,PG,NaN,Excel Entertainment,Based on Book/Short Story,Drama,Contemporary Fiction,NaN,17.0,6.0,531.0


In [ ]:
movies.info()


<class 'pandas.DataFrame'>
RangeIndex: 3201 entries, 0 to 3200
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   Title                   3200 non-null   str           
 1   US Gross                3194 non-null   float64       
 2   Worldwide Gross         3194 non-null   float64       
 3   US DVD Sales            564 non-null    float64       
 4   Production Budget       3200 non-null   float64       
 5   Release Date            3201 non-null   datetime64[us]
 6   MPAA Rating             2596 non-null   str           
 7   Running Time min        1209 non-null   float64       
 8   Distributor             2969 non-null   str           
 9   Source                  2836 non-null   str           
 10  Major Genre             2926 non-null   str           
 11  Creative Type           2755 non-null   str           
 12  Director                1870 non-null   str           
 13 

In [ ]:
overview = pd.DataFrame({
    "dtype": movies.dtypes,
    "non_null": movies.notna().sum(),
    "unique": movies.nunique(),
    "example": movies.apply(lambda col: col.dropna().iloc[0]),
})
overview

,dtype,non_null,unique,example
Title,str,3200,3176,The Land Girls
US Gross,float64,3194,3060,146083.0
Worldwide Gross,float64,3194,3074,146083.0
US DVD Sales,float64,564,564,3479242.0
Production Budget,float64,3200,381,8000000.0
Release Date,datetime64[us],3201,1600,1998-06-12 00:00:00
MPAA Rating,str,2596,7,R
Running Time min,float64,1209,109,104.0
Distributor,str,2969,174,Gramercy
Source,str,2836,18,Original Screenplay


In [ ]:
with pd.option_context("display.float_format", "{:,.2f}".format):
    display(movies.describe())

,US Gross,Worldwide Gross,US DVD Sales,Production Budget,Release Date,Running Time min,Rotten Tomatoes Rating,IMDB Rating,IMDB Votes
count,"3,194.00","3,194.00",564.00,"3,200.00",3201,"1,209.00","2,321.00","2,988.00","2,988.00"
mean,"44,002,085.16","85,343,400.14","34,901,546.82","31,069,171.45",1998-12-06 23:41:06.354264,110.19,54.34,6.28,"29,908.64"
min,0.00,0.00,"618,454.00",218.00,1928-12-31 00:00:00,46.00,1.00,1.40,18.00
25%,"5,493,221.25","8,031,285.25","9,906,210.75","6,575,000.00",1996-09-20 00:00:00,95.00,30.00,5.60,"4,828.50"
50%,"22,019,465.50","31,168,926.50","20,331,557.50","20,000,000.00",2001-12-14 00:00:00,107.00,55.00,6.40,"15,106.00"
75%,"56,091,761.50","97,283,797.00","37,794,215.75","42,000,000.00",2005-12-16 00:00:00,121.00,80.00,7.20,"35,810.50"
max,"760,167,650.00","2,767,891,499.00","352,582,053.00","300,000,000.00",2046-12-31 00:00:00,222.00,100.00,9.20,"519,541.00"
std,"62,555,311.39","149,947,342.89","45,895,121.60","35,585,913.44",NaN,20.17,28.08,1.25,"44,937.58"


In [ ]:
movies.describe(include=["object", "string"])

,Title,MPAA Rating,Distributor,Source,Major Genre,Creative Type,Director
count,3200,2596,2969,2836,2926,2755,1870
unique,3176,7,174,18,12,9,550
top,"20,000 Leagues Under the Sea",R,Warner Bros.,Original Screenplay,Drama,Contemporary Fiction,Steven Spielberg
freq,2,1194,318,1536,789,1453,23


In [ ]:
for col in ["MPAA Rating", "Major Genre", "Source", "Creative Type"]:
    display(movies[col].value_counts(dropna=False))

MPAA Rating
R            1194
PG-13         865
NaN           605
PG            354
Not Rated      94
G              79
NC-17           8
Open            2
Name: count, dtype: int64

Major Genre
Drama                  789
Comedy                 675
Action                 420
NaN                    275
Adventure              274
Thriller/Suspense      239
Horror                 219
Romantic Comedy        137
Musical                 53
Documentary             43
Western                 36
Black Comedy            36
Concert/Performance      5
Name: count, dtype: int64

Source
Original Screenplay              1536
Based on Book/Short Story         657
NaN                               365
Based on Real Life Events         186
Remake                            126
Based on TV                       106
Based on Comic/Graphic Novel       82
Based on Play                      55
Based on Game                      22
Traditional/Legend/Fairytale       16
Based on Magazine Article          10
Based on Musical/Opera              9
Based on Short Film                 8
Spin-Off                            8
Based on Factual Book/Article       6
Disney Ride                         4
Compilation                         3
Based on Toy                        1
Musical Group Movie                 1
Name: count, dtype: int64

Creative Type
Contemporary Fiction       1453
NaN                         446
Historical Fiction          350
Fantasy                     265
Science Fiction             243
Dramatization               204
Kids Fiction                141
Factual                      49
Super Hero                   49
Multiple Creative Types       1
Name: count, dtype: int64

### What the data is

**One row is one film** (a release). The table has 3,201 rows and 16 columns. Some titles appear twice (24 repeated titles), but the ones checked appear to be remakes with different release dates, so they are separate rows.

**Columns we will use:**
| Column | Type | Meaning | Why |
|---|---|---|---|
| Title | text | film title | to identify rows |
| Release Date | datetime64 | release date | year, decades, and spotting date errors |
| Production Budget | number | budget (currency not stated) | budget versus earnings |
| US Gross, Worldwide Gross | number | box office (currency not stated) | budget versus earnings, genre comparison |
| Rotten Tomatoes Rating | number, 1–100 | critics' score | critics versus audiences |
| IMDB Rating | number, 1.4–9.2 | audience score | critics versus audiences, genre comparison |
| IMDB Votes | number | number of votes | to judge how reliable a rating is |
| Major Genre | text, 12 values | genre | merge key, genre comparison |

**Columns we will probably not use:** US DVD Sales, Running Time min, Director, Distributor, Source, Creative Type, and MPAA Rating. Most of them have a lot of missing values.

**Not stated in the data:** the currency and year of the money columns, and the date on which the ratings were collected.

#### Tentative questions

These are the questions we plan to answer. They are tentative: cleaning and missing values may change how many films we can use for each one, and we may adjust the wording after that.

1. **Budget and earnings.** Do films with a bigger production budget earn more at the box office?
   *(Production Budget, Worldwide Gross)*
2. **Critics and audiences.** Do critics and audiences rate films alike?
   *(Rotten Tomatoes Rating, IMDB Rating; the two scales must be made comparable first)*
3. **Genres.** How do the genre groups differ in ratings and in money?
   *(Genre group from the second table, ratings, gross)*


## Part 2: Clean the data

### Problem 1: release dates in the future

**Problem.** There are several movies in this dataset with release dates in the future. But, since the site for vega-datasets doesn't identify either the source of this dataset. nor the date when it was made, we'll have to do some manual checking. Examination of a few entries in the dataset, and checking them against the IMDB, shows that movies with release dates before 2010 appear to have correct release dates. So we'll check the moviess with release dates after year 2010 to determine the most likely cut-off date.

In [ ]:
# Columns that might help us identify a particular movie
ID_COLS = ["Title", "Release Date", "Director", "Running Time min", "Distributor"]

def films_in_years(df, first=None, last=None, cols=ID_COLS):
    """Films released from year `first` to year `last` (both inclusive), oldest first.
    Leave one out to get 'from first onward' or 'up to last'."""
    year = df["Release Date"].dt.year
    mask = pd.Series(True, index=df.index)
    if first is not None:
        mask &= year >= first
    if last is not None:
        mask &= year <= last
    return df.loc[mask, cols].sort_values("Release Date")

print(len(films_in_years(movies, last=2010)))
display(films_in_years(movies, first=2011, last=2014))
display(films_in_years(movies, first=2015))


3177


,Title,Release Date,Director,Running Time min,Distributor
2658,Restless,2011-01-28,NaN,NaN,Sony Pictures
2967,The Adventures of Tintin: Secret of the Unicorn,2011-12-23,Steven Spielberg,NaN,Paramount Pictures


,Title,Release Date,Director,Running Time min,Distributor
120,The Birth of a Nation,2015-02-08,NaN,NaN,NaN
467,Intolerance,2016-09-05,NaN,NaN,NaN
26,"20,000 Leagues Under the Sea",2016-12-24,NaN,NaN,NaN
15,Over the Hill to the Poorhouse,2020-09-17,NaN,NaN,NaN
102,The Big Parade,2025-01-01,King Vidor,NaN,MGM
85,Ben-Hur,2025-12-30,NaN,NaN,MGM
1028,Wings,2027-08-12,NaN,NaN,NaN
822,She Done Him Wrong,2033-02-09,NaN,NaN,NaN
33,42nd Street,2033-03-09,NaN,NaN,Warner Bros.
495,King Kong (1933),2033-04-07,NaN,100.0,NaN


Checking both IMDB and Wikipedia, we can see that "The Adventures of Tintin" was released in 2011, and so was "Restless" distributed by Sony. We can also find "The Birth of a Nation" from 1915, and "Intolerance" from 1916. As to the "20,000 Leagues Under the Sea", there have been numerous productions of this story, and one of them was released in 1916, and none was released in 2016. From this we conclude that the most likely cut-off year is 2011, and everything in the dataset with the release date after this year is likely 100 years off. The fact that there are no movies in this dataset with release year between 2012 and 2014 can also be interpreted as supporting the conclusion.

#### Dates in the future

The data source [vega-datasets](https://vega.github.io/vega-datasets/datasets/movies/) doesn't say when the data was collected. To decide where the cut-off really is, we listed the films dated 2010-2012 and 2011-2016 with director, running time, and distributor, and checked a selection by hand against IMDB and Wikipedia.

- Films up to 2010 (3177 rows): four movies checked ("Firestorm", "Star Trek: Generations", "Forrest Gump", "K-PAX") -- all four appear with the correct release date.
- 2011 (2 rows): both movies appear to have correct release dates.
- 2015 and later (22 rows): four movies checked ("The Birth of a Nation", "Intolerance", "20,000 Leagues Under the Sea", "Duel in the Sun") -- all four apear to have the release date 100 years off.

**Decision:** subtract 100 years from the 22 rows dated 2015 or later; leave the rows with release date before 2011 unchanged.

In [ ]:
century = movies["Release Date"].dt.year >= 2015
movies.loc[century, "Release Date"] = movies.loc[century, "Release Date"] - pd.DateOffset(years=100)
print(century.sum(), movies["Release Date"].min(), movies["Release Date"].max())

22 1915-02-08 00:00:00 2011-12-23 00:00:00


### Problem 2: garbled accents in titles

Some titles contain characters that appear to be misplaced (e.g. `È` in "LÈon"), which looks like text-encoding damage. This problem was not visible in the summaries of Part 1; it was found by manually examining the `Title` values. The next cells isolate the titles with non-ASCII characters and test one explanation for the damage.

In [ ]:
non_ascii = movies["Title"].str.contains(r"[^\x00-\x7f]", na=False)
print(non_ascii.sum())
movies.loc[non_ascii, ["Title", "Release Date", "Distributor"]]

20


,Title,Release Date,Distributor
40,AstÈrix aux Jeux Olympiques,2008-07-04,Alliance
113,De battre mon coeur s'est arrÍtÈ,2005-07-01,WellSpring
137,Les BronzÈs 3: amis pour la vie,2006-02-01,NaN
316,Per qualche dollaro in pi˘,1967-05-10,MGM
460,I Love You Ö Don't Touch Me!,1998-02-20,MGM
534,Alien³,1992-05-22,20th Century Fox
648,The Naked Gun 2Ω: The Smell of Fear,1991-06-28,Paramount Pictures
729,LÈon,1994-11-18,Sony Pictures
763,Ri¢hie Ri¢h,1994-12-21,Warner Bros.
1163,Le Fabuleux destin d'AmÈlie Poulain,2001-11-02,Miramax


In [ ]:
chars = sorted(set("".join(movies.loc[non_ascii, "Title"])) - set(map(chr, range(128))))

rows = []
for c in chars:
    try:
        b = c.encode("mac_roman")          # the byte this character would have in Mac OS Roman
        rows.append((c, b.hex(), b.decode("cp1252", errors="replace"), b.decode("latin-1")))
    except UnicodeEncodeError:             # the character does not exist in Mac OS Roman
        rows.append((c, "-", "(not in Mac Roman)", "-"))

pd.DataFrame(rows, columns=["char", "mac_roman_byte", "as_cp1252", "as_latin1"])

,char,mac_roman_byte,as_cp1252,as_latin1
0,¢,a2,¢,¢
1,³,-,(not in Mac Roman),-
2,·,e1,á,á
3,È,e9,é,é
4,Ë,e8,è,è
5,Í,ea,ê,ê
6,Î,eb,ë,ë
7,Ö,85,…,
8,Ù,f4,ô,ô
9,Û,f3,ó,ó


20 titles contain non-ASCII characters (two cells above). Most of the characters in the table (`È`, `Í`, `Ö`, `Ω`, ...) are not plausible in a that position in a film title, but each of them turns into a plausible character (`é`, `ê`, `…`, `½`) when the Mac OS Roman byte it corresponds to is read as Windows-1252. This is consistent with text that was written as Windows-1252 and read as Mac OS Roman. The cause is inferred from this pattern and is not documented in the source.

Windows-1252 was chosen over Latin-1 because they differ for byte `85` (`Ö` in the data): Windows-1252 gives `…`, Latin-1 gives an invisible control character (last column).

Two characters do not follow the pattern: `³` does not exist in Mac OS Roman, and `¢` is unchanged by the conversion.

**Next:** the titles are re-encoded with Mac OS Roman and decoded with Windows-1252. Titles that cannot be re-encoded are left as they are.

In [ ]:
def fix_text(s):
    """Repair text that was written as Windows-1252 but read as Mac OS Roman."""
    try:
        # encoding with mac_roman recovers the original bytes;
        # decoding with cp1252 reads them the way they were written
        return s.encode("mac_roman").decode("cp1252")
    except UnicodeError:
        return s    # characters that Mac Roman cannot encode: left unchanged

before = movies["Title"].copy()
movies["Title"] = movies["Title"].map(fix_text, na_action="ignore")
changed = movies["Title"].fillna("") != before.fillna("")

print(changed.sum())
print(movies.loc[non_ascii & ~changed, "Title"].tolist())
pd.DataFrame({"before": before[changed], "after": movies.loc[changed, "Title"]})

18
['Alien³', 'Ri¢hie Ri¢h']


,before,after
40,AstÈrix aux Jeux Olympiques,Astérix aux Jeux Olympiques
113,De battre mon coeur s'est arrÍtÈ,De battre mon coeur s'est arrêté
137,Les BronzÈs 3: amis pour la vie,Les Bronzés 3: amis pour la vie
316,Per qualche dollaro in pi˘,Per qualche dollaro in più
460,I Love You Ö Don't Touch Me!,I Love You … Don't Touch Me!
648,The Naked Gun 2Ω: The Smell of Fear,The Naked Gun 2½: The Smell of Fear
729,LÈon,Léon
1163,Le Fabuleux destin d'AmÈlie Poulain,Le Fabuleux destin d'Amélie Poulain
1168,AimÈe & Jaguar,Aimée & Jaguar
1251,La mala educaciÛn,La mala educación


18 titles were changed. All 18 results are readable titles (e.g. `LÈon` --> `Léon`, `La mala educaciÛn` --> `La mala educación`). The 2 titles that were not changed (`Alien³`, `Ri¢hie Ri¢h`) are correct as they stand, which agrees with the table above. The titles without non-ASCII characters were not touched.

### Problem 3: zeros in the gross columns

Identified in Part 1: `.describe()` shows a minimum of 0 for both `US Gross` and `Worldwide Gross`. A gross of exactly 0 is suspect: it may mean "no figure available" and not "earned nothing".

The next cells count the zeros, show how they overlap between the two columns, and show when the films were released.

In [ ]:
# Count zeros
gross = ["US Gross", "Worldwide Gross"]
pd.DataFrame({"zeros": (movies[gross] == 0).sum(), "missing": movies[gross].isna().sum()})

,zeros,missing
US Gross,66,7
Worldwide Gross,47,7


In [ ]:
# Overlap between columns
us0 = movies["US Gross"] == 0
ww0 = movies["Worldwide Gross"] == 0
pd.crosstab(us0, ww0, rownames=["US Gross is 0"], colnames=["Worldwide Gross is 0"])

Worldwide Gross is 0,False,True
US Gross is 0,,
False,3135,0
True,19,47


In [ ]:
any0 = us0 | ww0
decade = movies["Release Date"].dt.year // 10 * 10
by_decade = any0.groupby(decade).agg(zero_rows="sum", no_of_films="size")
by_decade["share"] = (by_decade["zero_rows"] / by_decade["no_of_films"]).round(2)
by_decade

,zero_rows,no_of_films,share
Release Date,,,
1910,0,3,0.00
1920,0,6,0.00
1930,1,15,0.07
1940,1,19,0.05
1950,2,38,0.05
1960,8,71,0.11
1970,1,100,0.01
1980,1,256,0.00
1990,1,769,0.00


In [ ]:
movies.loc[any0 & (movies["Release Date"].dt.year >= 2010),
           ["Title", "Release Date", "Production Budget", "US Gross", "Worldwide Gross"]]

,Title,Release Date,Production Budget,US Gross,Worldwide Gross
1907,How Do You Know?,2010-12-17,90000000.0,0.0,0.0
2402,Shinjuku Incident,2010-02-05,25000000.0,0.0,0.0
2658,Restless,2011-01-28,15000000.0,0.0,0.0
2967,The Adventures of Tintin: Secret of the Unicorn,2011-12-23,130000000.0,0.0,0.0


- `US Gross` contains 66 zeros and `Worldwide Gross` 47 (7 values are missing in each column).
- All 47 rows with a worldwide gross of 0 also have a US gross of 0. In a further 19 rows, the US gross is 0 while the worldwide gross is above 0. If the worldwide gross includes the US gross (the data does not say), a US zero next to a positive worldwide figure cannot be a measured value, so in these rows the zero stands for "no figure".
- In total, 66 rows have a zero in at least one column. Zeros occur in every decade from the 1930s onward; 47 of the 66 are from the 2000s, the decade with the most films (1,830 films).
- 4 of the zero rows are dated 2010 or later; 2 of them are dated 2011 -- the year of collection (see Problem 1).

**Decision:** the zeros are replaced by missing values in both columns. A zero cannot be told apart from "no figure" in this data, and it would pull down means and medians. Keeping the zeros was rejected because of the 19 rows above.

In [ ]:
na_before = movies[gross].isna().sum()
movies[gross] = movies[gross].replace(0, np.nan)
na_after = movies[gross].isna().sum()

print((movies[gross] == 0).sum().to_dict())
pd.DataFrame({"missing before": na_before, "missing after": na_after, "changed": na_after - na_before})

{'US Gross': 0, 'Worldwide Gross': 0}


,missing before,missing after,changed
US Gross,7,73,66
Worldwide Gross,7,54,47


66 values in `US Gross` and 47 in `Worldwide Gross` were changed from 0 to missing, and no zeros remain. `US Gross` now has 73 missing values and `Worldwide Gross` 54 (7 each were missing before). Rows without a gross are left out only of the calculations that need it.

### Further checks: Duplicates and extra spaces

In [ ]:
text_cols = movies.select_dtypes(include=["object", "string"]).columns
padded = pd.Series({col: (movies[col].dropna() != movies[col].dropna().str.strip()).sum()
                    for col in text_cols})
padded

Title            0
MPAA Rating      0
Distributor      0
Source           0
Major Genre      0
Creative Type    0
Director         0
dtype: int64

In [ ]:
print(movies.duplicated().sum())
print(movies.duplicated(subset=["Title", "Release Date"]).sum())
print(movies["Title"].duplicated(keep=False).sum())

0
0
48


**Leading and trailing spaces.** All 7 text columns were checked, and no value has spaces at the start or end (0 rows in every column).

**Duplicates.** No row is repeated in every column (0 rows), and no title occurs twice with the same release date (0 rows). 48 rows share their title with another row, but with a different release date, so they represent different releases and are kept.

**Result:** nothing needs to be done for either check.

## Part 3: Missing values

The missing values are counted in every column, then examined to see where they occur and what they may mean. Decisions are made for the columns that matter for the questions.

In [ ]:
# Count missing values per column
missing = pd.DataFrame({
    "missing": movies.isna().sum(),
    "percent": (movies.isna().mean() * 100).round(1),
}).sort_values("missing", ascending=False)

missing

,missing,percent
US DVD Sales,2637,82.4
Running Time min,1992,62.2
Director,1331,41.6
Rotten Tomatoes Rating,880,27.5
MPAA Rating,605,18.9
Creative Type,446,13.9
Source,365,11.4
Major Genre,275,8.6
Distributor,232,7.2
IMDB Rating,213,6.7


In [ ]:
# Check where the missing values occur
# Each cell in the result is the share of missing values for that column in that decade
decade = movies["Release Date"].dt.year // 10 * 10
with_missing = movies.columns[movies.isna().any()]
share = movies[with_missing].isna().groupby(decade).mean().round(2)

display(share.T)
# Film counts per decade
decade.value_counts().sort_index().to_frame("films").T

Release Date,1910,1920,1930,1940,1950,1960,1970,1980,1990,2000,2010
Title,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
US Gross,0.33,0.33,0.07,0.05,0.05,0.11,0.05,0.00,0.00,0.03,0.04
Worldwide Gross,0.33,0.33,0.07,0.05,0.05,0.10,0.05,0.00,0.00,0.02,0.04
US DVD Sales,1.00,1.00,1.00,0.95,0.97,1.00,0.96,0.98,1.00,0.71,0.70
Production Budget,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00
MPAA Rating,1.00,1.00,0.87,0.95,0.87,0.87,0.82,0.88,0.15,0.02,0.15
Running Time min,1.00,1.00,0.80,1.00,1.00,1.00,1.00,0.98,0.60,0.52,0.71
Distributor,1.00,0.50,0.27,0.58,0.50,0.45,0.35,0.08,0.03,0.04,0.07
Source,0.33,0.50,0.40,0.63,0.61,0.58,0.46,0.17,0.07,0.06,0.23
Major Genre,0.33,0.50,0.33,0.68,0.66,0.55,0.44,0.07,0.05,0.04,0.15


Release Date,1910,1920,1930,1940,1950,1960,1970,1980,1990,2000,2010
films,3,6,15,19,38,71,100,256,769,1830,94


In [ ]:
# Check missing values overlaps
both = movies[["Rotten Tomatoes Rating", "IMDB Rating"]].notna().all(axis=1)
rt_missing_imdb_present = movies["Rotten Tomatoes Rating"].isna() & movies["IMDB Rating"].notna()
imdb_together = (movies["IMDB Rating"].isna() == movies["IMDB Votes"].isna()).all()

print("both ratings present:", both.sum())
print("Rotten Tomatoes missing, IMDB present:", rt_missing_imdb_present.sum())
print("IMDB Rating and IMDB Votes always missing together:", imdb_together)

both ratings present: 2260
Rotten Tomatoes missing, IMDB present: 728
IMDB Rating and IMDB Votes always missing together: True


Every column except `Release Date` has missing values. Four columns are missing in more than a quarter of the rows: `US DVD Sales` (82.4%), `Running Time min` (62.2%), `Director` (41.6%) and `Rotten Tomatoes Rating` (27.5%). `IMDB Rating` and `IMDB Votes` are missing in exactly the same 213 rows.

**US DVD Sales.** The value is missing for 95–100% of the films in every decade before 2000, but also for 71% of the films in the 2000s and 70% in the 2010s. The figure was therefore recorded for only some films, and a missing value cannot be read as zero sales. A likely part of the explanation for the old films is that DVDs did not exist yet; the reason for the gaps in later decades is unknown.  
*Decision: the column is dropped. It is mostly empty, it is not needed for the questions, and filling it would be guessing.*

**Rotten Tomatoes Rating.** The missing values are spread over all decades (18–41% of the films in each), so this is not only a problem of old films. 728 films have an IMDB rating but no Rotten Tomatoes rating, and 2,260 films have both. The reason is unknown; the source does not say how the films were matched.  
*Decision: the values are kept as missing and are not filled in. Question 2 (critics and audiences) uses only the 2,260 films that have both ratings, and the answer applies only to those films.*

**IMDB Rating and IMDB Votes.** Both are missing for 0–20% of the films in every decade from the 1920s onward (14% in the 2010s), and for 67% in the 1910s, which have only 3 films. The 213 films without an IMDB rating also drop out of question 2, in addition to the Rotten Tomatoes gaps counted above.  
*Decision: the values are kept as missing.*

**Major Genre.** The genre is missing for 33–68% of the films before 1980, but only 4–7% from the 1980s to the 2000s (15% in the 2010s). The genre comparison will therefore cover older films less well than newer ones.  
*Decision: the values are kept as missing here. Rows without a genre cannot be matched to the genre groups; how to label them is decided at the merge (Part 4).*

**Creative Type (446 missing, 13.9%).** The pattern is similar to `Major Genre`: 63–100% missing in every decade before 1980, 8–12% in the 1980s to the 2000s, and 26% in the 2010s. The column is not used for the three questions and is left unchanged. If it is used later, the same limitation applies as for the genre.

**Gross columns.** `US Gross` (73) and `Worldwide Gross` (54) are missing after Problem 3 replaced the zeros.  
*Decision: kept as missing (see Problem 3).*

**Other columns.** `Running Time min`, `Director`, `MPAA Rating`, `Source` and `Distributor` are not used for the three questions and are left unchanged. `Title` and `Production Budget` are missing in one row each, which is kept; the row simply drops out of any calculation that needs the missing value.

In [ ]:
shape_before = movies.shape
movies = movies.drop(columns="US DVD Sales")
print(shape_before, "->", movies.shape)

(3201, 16) -> (3201, 15)


`US DVD Sales` was dropped: the table went from 16 to 15 columns, and no rows were removed. All other missing values are kept as described above.